# FR and Baseline Transfer Analysis

This notebook is organized into setup, data loading, exploratory transfer summaries, matched statistical tests, sensitivity analysis, and optional meta-modeling sections.

Run the notebook from top to bottom when regenerating outputs. The meta-modeling sections are optional and depend on the prepared transfer data and core analysis helpers.


## Setup

Imports and global warning configuration.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
import numpy as np
from typing import Optional
import re
from pathlib import Path


import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Portable analysis paths
# ============================================================

import sys

cwd = Path.cwd().resolve()

# Supports launching Jupyter from either the repository root
# or directly from analysis/.
if (
    (cwd / "data").is_dir()
    and (cwd / "transfer_analysis_node_jackknife.py").is_file()
):
    ANALYSIS_DIR = cwd

elif (
    (cwd / "analysis" / "data").is_dir()
    and (
        cwd
        / "analysis"
        / "transfer_analysis_node_jackknife.py"
    ).is_file()
):
    ANALYSIS_DIR = cwd / "analysis"

else:
    raise FileNotFoundError(
        "Could not locate the analysis directory. "
        f"Current working directory: {cwd}"
    )

DATA_DIR = ANALYSIS_DIR / "data"
ANALYSIS_OUTPUT_DIR = ANALYSIS_DIR / "output"
FIGURES_DIR = ANALYSIS_OUTPUT_DIR / "figures"
STATISTICS_DIR = ANALYSIS_OUTPUT_DIR / "statistic"
TABLES_DIR = ANALYSIS_OUTPUT_DIR / "tables"

for directory in (
    FIGURES_DIR,
    STATISTICS_DIR,
    TABLES_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

print(f"Analysis directory: {ANALYSIS_DIR}")
print(f"Data directory:     {DATA_DIR}")
print(f"Output directory:   {ANALYSIS_OUTPUT_DIR}")


## Data Loading

Load clean-performance results, AA and CW-EOT pairwise transfer
results, and target-level oracle summaries from `data/`.

In [ ]:



def parse_origin_model_from_attack_path(path):
    """
    Convert an attack path into the full detector key expected downstream:
      imgnet_xception_ALL
      fr_resnet34_FS
    """
    path = Path(path)
    model_name = path.parent.name
    path_str = str(path).lower()

    if "no_pretrain" in path_str or "no_pretain" in path_str:
        return f"nopretrain_{model_name}"

    if "fr_pretrain" in path_str or "fr_pretain" in path_str:
        return f"fr_{model_name}"

    if "clip" in path_str:
        return model_name

    # Covers both:
    #   .../aa_th_aware/adv/adv/<model>/<attack>
    #   .../cw_eot_th_aware/imgnet/adv/<model>/<attack>
    return f"imgnet_{model_name}"


def infer_attack_type_from_attack_path(path):
    path_str = str(path).lower()

    if "cw_eot" in path_str or "eot" in path_str:
        return "cw_eot"

    if "autoattack" in path_str or "aa_th_aware" in path_str:
        return "aa"

    return "unknown"


def load_transfer_results(path, attack_type):
    df = pd.read_csv(path)

    required = {"target_model", "attack_path", "asr"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")

    # Preserve the model name originally stored in the CSV.
    if "origin_model" in df.columns:
        df["origin_model_raw"] = df["origin_model"]

    # Add pretraining prefix to obtain the complete source-model key.
    df["origin_model"] = df["attack_path"].apply(
        parse_origin_model_from_attack_path
    )
    df["attack_type"] = attack_type

    return df


def load_oracle_results(
    path,
    attack_type="aa_cw_eot_oracle",
    oracle_scope="strict",
):
    """
    Load the oracle computed jointly across AA and CW-EOT candidates.

    The file contains one aggregate row per target detector.
    """
    df = pd.read_csv(path)

    required = {"target_model", "asr"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")

    df["attack_type"] = attack_type
    df["oracle_scope"] = oracle_scope

    return df


def load_oracle_source_counts(
    path,
    attack_type="aa_cw_eot_oracle",
    oracle_scope="strict",
):
    """
    Load the source composition of the joint strict oracle.
    """
    df = pd.read_csv(path)

    df["attack_type"] = attack_type
    df["oracle_scope"] = oracle_scope

    # The current file already contains the complete origin key.
    if "origin_key" in df.columns:
        df["origin_model_display"] = df["origin_key"]
    elif "attack_path" in df.columns:
        df["origin_model_display"] = df["attack_path"].apply(
            parse_origin_model_from_attack_path
        )
    elif "origin_model" in df.columns:
        df["origin_model_display"] = df["origin_model"]
    else:
        df["origin_model_display"] = "unknown"

    if "attack_name" in df.columns:
        df["selected_attack_type"] = df["attack_name"].apply(
            infer_attack_type_from_attack_path
        )
    elif "attack_path" in df.columns:
        df["selected_attack_type"] = df["attack_path"].apply(
            infer_attack_type_from_attack_path
        )
    else:
        df["selected_attack_type"] = "unknown"

    return df


# ============================================================
# Clean performance
# ============================================================

# Clean-performance results for the released detector bank.
df_clean_perf = pd.read_csv(DATA_DIR / "all_model_results.csv")


# ============================================================
# Standard attack-specific transfer results
# ============================================================

df_aa = load_transfer_results(
    DATA_DIR / "transfer_results_aa_th_aware.csv",
    attack_type="aa",
)

df_eot = load_transfer_results(
    DATA_DIR / "transfer_results_cw_eot_th_aware.csv",
    attack_type="cw_eot",
)

df_transfer = pd.concat(
    [df_aa, df_eot],
    ignore_index=True,
)


# ============================================================
# Joint strict portfolio oracle across AA and CW-EOT
# ============================================================

df_oracle = load_oracle_results(
    DATA_DIR / "transfer_results_combined_strict_oracle.csv",
)

#df_oracle_source_counts = load_oracle_source_counts(
#    DATA_DIR / "transfer_results_combined_strict_oracle_source_counts.csv",
#)

#### Clean Perf

In [ ]:
import re
import numpy as np
import pandas as pd
from IPython.display import display

# Load if not already loaded
try:
    df_clean_perf
except NameError:
    df_clean_perf = pd.read_csv(DATA_DIR / "all_model_results.csv")


BACKBONE_ORDER = [
    "xception",
    "resnet34",
    "efficientnetb4",
    "deit",
    "vit",
    "swin",
]

BACKBONE_LABELS = {
    "xception": "Xception",
    "resnet34": "ResNet34",
    "efficientnetb4": "EffNet-B4",
    "deit": "DeiT-S",
    "vit": "ViT-B/16",
    "swin": "Swin-T",
}

PRETRAIN_ORDER = ["imgnet", "fr"]
TRAIN_FAMILY_ORDER = ["ALL", "FS", "FR", "EFS", "FE"]


def infer_pretrain(root_name, root_path):
    s = f"{root_name} {root_path}".lower()

    if "fr_pretrain" in s:
        return "fr"

    if "no_pretrain" in s or "nopretrain" in s:
        return "nopretrain"

    if str(root_name).lower() == "saved_models":
        return "imgnet"

    return "unknown"


def parse_backbone(model_name):
    s = str(model_name).lower()

    if s.startswith("xception"):
        return "xception"
    if s.startswith("resnet34"):
        return "resnet34"
    if s.startswith("efficientnetb4"):
        return "efficientnetb4"
    if s.startswith("deit"):
        return "deit"
    if s.startswith("vit"):
        return "vit"
    if s.startswith("swin"):
        return "swin"
    if s.startswith("clip"):
        return "clip"

    return "unknown"


def parse_train_family(model_name):
    s = str(model_name).upper()

    m = re.search(r"_TRAIN_(ALL|EFS|FE|FR|FS)_FF_", s)
    if m:
        return m.group(1)

    return "unknown"


def auc_to_unit(x):
    if pd.isna(x):
        return np.nan

    x = float(x)

    if x > 1.5:
        return x / 100.0

    return x


def fmt_auc(x):
    if pd.isna(x):
        return "--"

    return f"{x:.2f}"


# ------------------------------------------------------------
# Prepare dataframe
# ------------------------------------------------------------

clean = df_clean_perf.copy()

clean["pretrain"] = [
    infer_pretrain(r, p)
    for r, p in zip(clean["root_name"], clean["root_path"])
]

clean["backbone"] = clean["model_name"].apply(parse_backbone)
clean["train_family"] = clean["model_name"].apply(parse_train_family)

clean["model_key"] = (
    clean["pretrain"]
    + "_"
    + clean["backbone"]
    + "_"
    + clean["train_family"]
)

clean["auc_unit"] = clean["auc"].apply(auc_to_unit)

# Keep only test rows and the models used in the paper table
clean = clean[
    clean["phase"].eq("test")
    & clean["pretrain"].isin(PRETRAIN_ORDER)
    & clean["backbone"].isin(BACKBONE_ORDER)
    & clean["train_family"].isin(TRAIN_FAMILY_ORDER)
].copy()

# There are duplicate split rows from different eval jsons.
# They are usually identical; max is safe and avoids duplicates.
auc_lookup = (
    clean
    .groupby(["model_key", "split"])["auc_unit"]
    .max()
    .to_dict()
)


def get_auc(model_key, split):
    return auc_lookup.get((model_key, split), np.nan)


def make_clean_cell(pretrain, backbone, train_family):
    model_key = f"{pretrain}_{backbone}_{train_family}"

    ff_all = get_auc(model_key, "ALL_ff")
    cdf = get_auc(model_key, f"ALL_cdf")

    if train_family == "ALL":
        return rf"${fmt_auc(ff_all)}/{fmt_auc(cdf)}$"

    ff_id = get_auc(model_key, f"{train_family}_ff")
    return rf"${fmt_auc(ff_all)}({fmt_auc(ff_id)})/{fmt_auc(cdf)}$"


# ------------------------------------------------------------
# Build preview table
# ------------------------------------------------------------

rows = []

for backbone in BACKBONE_ORDER:
    row = {"Backbone": BACKBONE_LABELS[backbone]}

    for pretrain in PRETRAIN_ORDER:
        for train_family in TRAIN_FAMILY_ORDER:
            row[f"{pretrain}_{train_family}"] = make_clean_cell(
                pretrain=pretrain,
                backbone=backbone,
                train_family=train_family,
            )

    rows.append(row)

clean_table_df = pd.DataFrame(rows)
display(clean_table_df)


# ------------------------------------------------------------
# Generate LaTeX rows
# ------------------------------------------------------------

latex_rows = []

for _, row in clean_table_df.iterrows():
    latex_row = (
        f"{row['Backbone']:<10} "
        f"& {row['imgnet_ALL']} "
        f"& {row['imgnet_FS']} "
        f"& {row['imgnet_FR']} "
        f"& {row['imgnet_EFS']} "
        f"& {row['imgnet_FE']} "
        f"& {row['fr_ALL']} "
        f"& {row['fr_FS']} "
        f"& {row['fr_FR']} "
        f"& {row['fr_EFS']} "
        f"& {row['fr_FE']} \\\\"
    )

    latex_rows.append(latex_row)

latex_body = "\n".join(latex_rows)

print(latex_body)

with open(
    TABLES_DIR / "clean_performance_rows.tex",
    "w",
    encoding="utf-8",
) as f:
    f.write(latex_body)

print(
    "\nSaved rows to "
    f"{TABLES_DIR / 'clean_performance_rows.tex'}"
)

## Transfer Matrix Heatmaps

Build model-to-model ASR heatmaps for both AA and CW-EOT transfer results.


In [ ]:
def make_asr_cmap():
    return mcolors.LinearSegmentedColormap.from_list(
        "asr_map",
        [
            (0.0, "#c7e9c0"),
            (0.25, "#ffd92f"),
            (0.55, "#ff8c00"),
            (1.0, "#b30000"),
        ],
    )


def plot_transfer_heatmap(df, title, save_path=None, figsize=(20, 20)):
    plot_df = df.loc[
        ~df["target_model"].astype(str).str.contains(
            "clip", case=False, na=False
        )
        & ~df["origin_model"].astype(str).str.contains(
            "clip", case=False, na=False
        )
    ].copy()

    matrix_pct = (
        plot_df.pivot_table(
            index="target_model",
            columns="origin_model",
            values="asr",
            aggfunc="mean",
        )
        * 100
    )

    matrix_pct.index = (
        matrix_pct.index.astype(str)
        .str.replace(r"^fr_", "FaceRec_", regex=True)
    )

    matrix_pct.columns = (
        matrix_pct.columns.astype(str)
        .str.replace(r"^fr_", "FaceRec_", regex=True)
    )

    fig = plt.figure(figsize=figsize)
    sns.heatmap(
        matrix_pct,
        annot=True,
        fmt=".0f",
        cmap=make_asr_cmap(),
        vmin=0,
        vmax=100,
        cbar_kws={
            "label": "ASR (%)",
            "ticks": [0, 20, 40, 60, 80, 100],
        },
    )

    plt.xlabel("Origin model")
    plt.ylabel("Target model")
    plt.title(title)
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()

    return matrix_pct


matrix_aa_pct = plot_transfer_heatmap(
    df_aa,
    "AA transfer attack success rate (%)",
    save_path=FIGURES_DIR / "fig5_aa_transfer_matrix.pdf",
)

matrix_eot_pct = plot_transfer_heatmap(
    df_eot,
    "CW-EOT transfer attack success rate (%)",
    save_path=FIGURES_DIR / "fig6_cw_eot_transfer_matrix.pdf",
)

## Oracle Transfer Analysis

Compare strict and self-excluding oracle summaries for AA, CW-EOT,
and their combined portfolio. Oracle files contain one aggregate row
per target detector rather than an origin-by-target matrix.

In [ ]:
# ============================================================
# Model vulnerability summary
#
# Panels:
#   1. White-box ASR
#   2. Mean black-box ASR
#   3. Strict oracle ASR
#   4. Self-excluding oracle ASR
#
# Important:
#   - WB and mean BB values come only from the verified
#     threshold-aware transfer CSVs.
#   - *_individual.csv oracle files are never used for WB/BB.
#   - The cell requires AA, CW-EOT, and Combined oracle summaries
#     for both strict and self-excluding rules.
#
# This assumes load_transfer_results() is already defined in the
# notebook, using the same version that gave the correct WB values.
# ============================================================

from pathlib import Path
import ast
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

RESULTS_DIR = DATA_DIR
RECURSIVE_SEARCH = False
EXCLUDE_CLIP = True
SAVE_FIGURE = True

OUTPUT_DIR = FIGURES_DIR / "vulnerability_plots"
OUTPUT_STEM = OUTPUT_DIR / "model_vulnerability_summary"

PAIRWISE_PATHS = {
    "aa": RESULTS_DIR / "transfer_results_aa_th_aware.csv",
    "cw_eot": RESULTS_DIR / "transfer_results_cw_eot_th_aware.csv",
}

ATTACK_ORDER = ["aa", "cw_eot", "combined"]

ATTACK_LABELS = {
    "aa": "AA",
    "cw_eot": "CW-EOT",
    "combined": r"AA $\cup$ CW-EOT",
}

PRETRAIN_ORDER = ["imgnet", "fr"]

PRETRAIN_LABELS = {
    "imgnet": "ImageNet",
    "fr": "FR",
}

PALETTE = {
    "imgnet": "#4C78A8",
    "fr": "#E45756",
}

METRIC_ORDER = [
    "White-box ASR",
    "Mean black-box ASR",
    "Strict oracle ASR",
    "Self-excluding oracle ASR",
]

REQUIRED_ORACLES = {
    (attack, rule)
    for rule in ("strict", "self_excluding")
    for attack in ATTACK_ORDER
}


# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

if "load_transfer_results" not in globals():
    raise NameError(
        "load_transfer_results() is not defined. Run the cell that defines "
        "your verified loader before running this plotting cell."
    )

if not RESULTS_DIR.exists():
    raise FileNotFoundError(
        f"Results directory does not exist:\n{RESULTS_DIR.resolve()}"
    )

for attack, path in PAIRWISE_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing authoritative {attack} transfer file:\n{path.resolve()}"
        )


# ------------------------------------------------------------
# Load authoritative threshold-aware pairwise results
# ------------------------------------------------------------

df_aa = load_transfer_results(
    str(PAIRWISE_PATHS["aa"]),
    attack_type="aa",
)

df_eot = load_transfer_results(
    str(PAIRWISE_PATHS["cw_eot"]),
    attack_type="cw_eot",
)

PAIRWISE_DATA = {
    "aa": df_aa.copy(),
    "cw_eot": df_eot.copy(),
}


# ------------------------------------------------------------
# General helpers
# ------------------------------------------------------------

def normalized_stem(path):
    """
    Lowercase filename stem and remove copy suffixes such as:
        file(1).csv
        file (2).csv
    """
    stem = Path(path).stem.lower().strip()
    return re.sub(r"\s*\(\d+\)$", "", stem)


def convert_asr_to_unit_interval(values):
    """
    Accept ASR expressed either as [0, 1] or [0, 100].
    """
    values = pd.to_numeric(values, errors="coerce")

    if values.notna().any() and values.max() > 1.5:
        values = values / 100.0

    return values


def parse_pretraining(model_name):
    name = str(model_name).lower()

    if name.startswith("fr_"):
        return "fr"

    if name.startswith("imgnet_") or name.startswith("imagenet_"):
        return "imgnet"

    return "unknown"


def parse_boolean_series(series, column_name):
    """
    Convert common boolean encodings to a proper Boolean series.
    """
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(series):
        converted = series.map({
            1: True,
            0: False,
            1.0: True,
            0.0: False,
        })
    else:
        converted = (
            series.astype(str)
            .str.strip()
            .str.lower()
            .map({
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            })
        )

    if converted.isna().any():
        invalid = series[converted.isna()].drop_duplicates().tolist()
        raise ValueError(
            f"Could not interpret all values in {column_name!r} as "
            f"booleans. Unrecognized values include: {invalid[:10]}"
        )

    return converted.astype(bool)


# ------------------------------------------------------------
# Pairwise summarization
# ------------------------------------------------------------

def summarize_pairwise_data(df, attack):
    """
    Produce one WB value and one mean-BB value per target model.

    The function prefers the is_whitebox definition produced by the
    verified load_transfer_results() function. It falls back to exact
    source/target equality only if is_whitebox is unavailable.
    """
    d = df.copy()

    required = {"target_model", "asr"}
    missing = required.difference(d.columns)

    if missing:
        raise ValueError(
            f"{ATTACK_LABELS[attack]} pairwise data is missing "
            f"columns: {sorted(missing)}"
        )

    d["target_model"] = d["target_model"].astype(str)
    d["asr"] = convert_asr_to_unit_interval(d["asr"])
    d = d.dropna(subset=["target_model", "asr"])

    source_col = None

    for candidate in ("origin_key", "origin_model", "source_model"):
        if candidate in d.columns:
            source_col = candidate
            d[source_col] = d[source_col].astype(str)
            break

    if EXCLUDE_CLIP:
        keep = ~d["target_model"].str.contains(
            "clip",
            case=False,
            na=False,
        )

        if source_col is not None:
            keep &= ~d[source_col].str.contains(
                "clip",
                case=False,
                na=False,
            )

        d = d[keep].copy()

    if "is_whitebox" in d.columns:
        d["is_whitebox"] = parse_boolean_series(
            d["is_whitebox"],
            "is_whitebox",
        )
        wb_definition = "verified is_whitebox column"

    elif source_col is not None:
        d["is_whitebox"] = (
            d[source_col].astype(str)
            == d["target_model"].astype(str)
        )
        wb_definition = f"{source_col} == target_model"

    else:
        raise ValueError(
            f"{ATTACK_LABELS[attack]} data has neither is_whitebox "
            "nor a recognizable source-model column."
        )

    wb_rows = d[d["is_whitebox"]].copy()
    bb_rows = d[~d["is_whitebox"]].copy()

    if wb_rows.empty:
        raise ValueError(
            f"No white-box rows were identified for "
            f"{ATTACK_LABELS[attack]}."
        )

    if bb_rows.empty:
        raise ValueError(
            f"No black-box rows were identified for "
            f"{ATTACK_LABELS[attack]}."
        )

    wb = (
        wb_rows
        .groupby("target_model", as_index=False)
        .agg(
            value=("asr", "mean"),
            row_count=("asr", "size"),
        )
        .assign(
            attack=attack,
            metric="White-box ASR",
            oracle_rule="not_applicable",
            n_sources=1,
        )
    )

    if source_col is not None:
        bb = (
            bb_rows
            .groupby("target_model", as_index=False)
            .agg(
                value=("asr", "mean"),
                n_sources=(source_col, "nunique"),
                row_count=("asr", "size"),
            )
        )
    else:
        bb = (
            bb_rows
            .groupby("target_model", as_index=False)
            .agg(
                value=("asr", "mean"),
                row_count=("asr", "size"),
            )
            .assign(n_sources=np.nan)
        )

    bb = bb.assign(
        attack=attack,
        metric="Mean black-box ASR",
        oracle_rule="not_applicable",
    )

    print(f"\n{ATTACK_LABELS[attack]} pairwise results")
    print(f"  Source: {PAIRWISE_PATHS[attack].name}")
    print(f"  WB definition: {wb_definition}")
    print(f"  WB target models: {wb['target_model'].nunique()}")
    print(
        "  WB minimum / median / maximum: "
        f"{100 * wb['value'].min():.2f}% / "
        f"{100 * wb['value'].median():.2f}% / "
        f"{100 * wb['value'].max():.2f}%"
    )
    print(
        "  Mean-BB minimum / median / maximum: "
        f"{100 * bb['value'].min():.2f}% / "
        f"{100 * bb['value'].median():.2f}% / "
        f"{100 * bb['value'].max():.2f}%"
    )

    return pd.concat([wb, bb], ignore_index=True)


# ------------------------------------------------------------
# Oracle-file classification
# ------------------------------------------------------------

def inspect_oracle_schema(path):
    """
    Return True only for a target-level oracle summary.

    Pairwise/individual files and oracle-selection files are rejected.
    """
    try:
        columns = set(pd.read_csv(path, nrows=0).columns)
    except Exception:
        return False

    if not {"target_model", "asr"}.issubset(columns):
        return False

    origin_columns = {
        "origin_key",
        "origin_model",
        "source_model",
        "selected_origin",
        "selected_source",
    }

    if columns.intersection(origin_columns):
        return False

    return True


def infer_attack(path):
    """
    Determine whether an oracle summary is AA, CW-EOT, or Combined.
    """
    name = normalized_stem(path)

    if (
        "combined" in name
        or re.search(r"(^|_)aa_eot($|_)", name)
        or re.search(r"(^|_)aa_cw_eot($|_)", name)
    ):
        return "combined"

    if (
        "cw_eot" in name
        or re.search(r"(^|_)eot($|_)", name)
    ):
        return "cw_eot"

    if re.search(r"(^|_)aa($|_)", name):
        return "aa"

    return None


def load_rule_payload(value, csv_parent):
    """
    Parse an oracle-rule value when it is:
      - a dictionary;
      - a JSON/Python dictionary string;
      - a path to a JSON file.
    """
    if isinstance(value, dict):
        return value

    if value is None:
        return None

    if isinstance(value, float) and np.isnan(value):
        return None

    text = str(value).strip()

    if not text:
        return None

    possible_paths = [
        Path(text),
        csv_parent / text,
    ]

    for possible_path in possible_paths:
        if possible_path.is_file():
            try:
                with open(possible_path, "r", encoding="utf-8") as file:
                    return json.load(file)
            except Exception:
                pass

    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass

    return None


def classify_rule_payload(payload):
    """
    Classify a parsed rules dictionary.

    Strict:
      excludes self and at least one additional source category.

    Self-excluding:
      excludes only the source model itself.
    """
    if not isinstance(payload, dict):
        return None

    exclude_self = bool(
        payload.get(
            "exclude_self_origin",
            payload.get(
                "exclude_self",
                payload.get("exclude_source_model", False),
            ),
        )
    )

    strict_flags = [
        bool(payload.get("exclude_same_training_data", False)),
        bool(payload.get("exclude_same_architecture", False)),
        bool(payload.get("exclude_same_backbone", False)),
        bool(payload.get("exclude_same_train_family", False)),
        bool(payload.get("exclude_same_pretraining", False)),
        bool(payload.get("exclude_train_tags_as_source", [])),
        bool(payload.get("exclude_origin_models", [])),
    ]

    # Excluding CLIP alone should not turn a self-excluding rule into strict.
    origin_exclusions = payload.get("exclude_origin_models", [])

    if isinstance(origin_exclusions, str):
        origin_exclusions = [origin_exclusions]

    non_clip_origin_exclusions = [
        item
        for item in origin_exclusions
        if str(item).lower() != "clip"
    ]

    strict_flags[-1] = bool(non_clip_origin_exclusions)

    if exclude_self and any(strict_flags):
        return "strict"

    if exclude_self:
        return "self_excluding"

    return None


def infer_oracle_rule(path):
    """
    Infer the oracle rule from the filename first, then from rule metadata.
    """
    name = normalized_stem(path)

    self_tokens = (
        "self_excluded",
        "self_excluding",
        "exclude_self",
        "self_only",
    )

    strict_tokens = (
        "strict",
        "no_same_train",
        "no_same_arch",
        "exclude_same_train",
        "exclude_same_arch",
    )

    if any(token in name for token in self_tokens):
        return "self_excluding"

    if any(token in name for token in strict_tokens):
        return "strict"

    try:
        metadata = pd.read_csv(path, nrows=5)
    except Exception:
        return None

    possible_columns = (
        "oracle_rule_json",
        "oracle_rules_json",
        "rule_json",
        "rules_json",
        "oracle_rules",
    )

    for column in possible_columns:
        if column not in metadata.columns:
            continue

        for value in metadata[column].dropna().unique():
            payload = load_rule_payload(value, path.parent)
            rule = classify_rule_payload(payload)

            if rule is not None:
                return rule

    return None


def discover_oracle_files(results_dir):
    """
    Discover target-level oracle summaries only.

    If multiple copies are found for the same attack/rule combination,
    the most recently modified compatible file is selected.
    """
    results_dir = Path(results_dir)

    iterator = (
        results_dir.rglob("*")
        if RECURSIVE_SEARCH
        else results_dir.glob("*")
    )

    csv_files = [
        path
        for path in iterator
        if path.is_file() and path.suffix.lower() == ".csv"
    ]

    candidates = []

    ignored_tokens = (
        "oracle_selection",
        "source_count",
        "model_summary",
        "vulnerability_summary",
        "_individual",
    )

    for path in csv_files:
        name = normalized_stem(path)

        if "oracle" not in name:
            continue

        if any(token in name for token in ignored_tokens):
            continue

        if not inspect_oracle_schema(path):
            continue

        attack = infer_attack(path)
        rule = infer_oracle_rule(path)

        candidates.append({
            "path": path,
            "attack": attack,
            "rule": rule,
            "modified": path.stat().st_mtime,
        })

    if not candidates:
        raise FileNotFoundError(
            "No target-level oracle summary CSVs were recognized."
        )

    candidate_table = pd.DataFrame(candidates)

    print("\nRecognized oracle candidates")
    print(
        candidate_table[
            ["attack", "rule", "path"]
        ].to_string(index=False)
    )

    unclassified = candidate_table[
        candidate_table["attack"].isna()
        | candidate_table["rule"].isna()
    ]

    if not unclassified.empty:
        print("\nIgnored unclassified oracle candidates")

        for _, row in unclassified.iterrows():
            print(
                f"  {row['path'].name}: "
                f"attack={row['attack']}, rule={row['rule']}"
            )

    usable = candidate_table.dropna(
        subset=["attack", "rule"]
    ).copy()

    selected = {}

    for (attack, rule), group in usable.groupby(
        ["attack", "rule"]
    ):
        chosen = group.sort_values(
            "modified",
            ascending=False,
        ).iloc[0]["path"]

        selected[(attack, rule)] = chosen

        if len(group) > 1:
            rejected = [
                path.name
                for path in group["path"]
                if path != chosen
            ]

            warnings.warn(
                f"Multiple oracle summaries found for "
                f"{attack}/{rule}. Selected newest file "
                f"{chosen.name}; ignored {rejected}."
            )

    missing = REQUIRED_ORACLES.difference(selected)

    if missing:
        selected_text = "\n".join(
            f"  {attack}/{rule}: {path.name}"
            for (attack, rule), path in sorted(selected.items())
        )

        missing_text = "\n".join(
            f"  {attack}/{rule}"
            for attack, rule in sorted(missing)
        )

        raise FileNotFoundError(
            "The full four-panel figure requires six oracle summaries.\n\n"
            f"Selected:\n{selected_text or '  None'}\n\n"
            f"Missing:\n{missing_text}"
        )

    print("\nSelected oracle summaries")

    for rule in ("strict", "self_excluding"):
        for attack in ATTACK_ORDER:
            print(
                f"  {rule:>14} / "
                f"{ATTACK_LABELS[attack]:>8}: "
                f"{selected[(attack, rule)].name}"
            )

    return selected


# ------------------------------------------------------------
# Oracle summarization
# ------------------------------------------------------------

def summarize_oracle_file(path, attack, rule):
    d = pd.read_csv(path)

    required = {"target_model", "asr"}
    missing = required.difference(d.columns)

    if missing:
        raise ValueError(
            f"{path.name} is missing columns: {sorted(missing)}"
        )

    d["target_model"] = d["target_model"].astype(str)
    d["asr"] = convert_asr_to_unit_interval(d["asr"])
    d = d.dropna(subset=["target_model", "asr"])

    if EXCLUDE_CLIP:
        d = d[
            ~d["target_model"].str.contains(
                "clip",
                case=False,
                na=False,
            )
        ].copy()

    # A target-level oracle file should contain exactly one result per target.
    counts = d.groupby("target_model").size()
    duplicate_targets = counts[counts > 1]

    if not duplicate_targets.empty:
        raise ValueError(
            f"{path.name} has multiple rows for target models such as "
            f"{duplicate_targets.index[:5].tolist()}. The selected file "
            "may not be a target-level oracle summary."
        )

    metric = {
        "strict": "Strict oracle ASR",
        "self_excluding": "Self-excluding oracle ASR",
    }[rule]

    return (
        d[["target_model", "asr"]]
        .rename(columns={"asr": "value"})
        .assign(
            attack=attack,
            metric=metric,
            oracle_rule=rule,
            n_sources=np.nan,
            row_count=1,
        )
    )


# ------------------------------------------------------------
# Build complete model-level summary
# ------------------------------------------------------------

def build_summary(pairwise_data, oracle_files):
    parts = []

    for attack in ("aa", "cw_eot"):
        parts.append(
            summarize_pairwise_data(
                pairwise_data[attack],
                attack,
            )
        )

    for rule in ("strict", "self_excluding"):
        for attack in ATTACK_ORDER:
            parts.append(
                summarize_oracle_file(
                    oracle_files[(attack, rule)],
                    attack,
                    rule,
                )
            )

    summary = pd.concat(parts, ignore_index=True)

    summary["pretrain"] = (
        summary["target_model"]
        .map(parse_pretraining)
    )

    unknown_models = summary.loc[
        summary["pretrain"].eq("unknown"),
        "target_model",
    ].drop_duplicates()

    if not unknown_models.empty:
        warnings.warn(
            "Could not determine pretraining for models including: "
            f"{unknown_models.head(10).tolist()}"
        )

    summary["pretrain_label"] = (
        summary["pretrain"]
        .map(PRETRAIN_LABELS)
        .fillna(summary["pretrain"])
    )

    summary["attack_label"] = (
        summary["attack"]
        .map(ATTACK_LABELS)
        .fillna(summary["attack"])
    )

    summary["value_pct"] = 100 * summary["value"]

    invalid = summary[
        ~summary["value"].between(0, 1, inclusive="both")
    ]

    if not invalid.empty:
        raise ValueError(
            "Some ASR values remain outside [0, 1] after conversion."
        )

    return summary.sort_values(
        ["metric", "attack", "target_model"]
    ).reset_index(drop=True)


# ------------------------------------------------------------
# Coverage report
# ------------------------------------------------------------

def print_coverage(summary):
    coverage = (
        summary
        .groupby(
            ["metric", "attack"],
            observed=True,
        )
        .agg(
            models=("target_model", "nunique"),
            mean_pct=("value_pct", "mean"),
            median_pct=("value_pct", "median"),
            minimum_pct=("value_pct", "min"),
            maximum_pct=("value_pct", "max"),
        )
        .reset_index()
    )

    coverage["metric"] = pd.Categorical(
        coverage["metric"],
        categories=METRIC_ORDER,
        ordered=True,
    )

    coverage["attack"] = pd.Categorical(
        coverage["attack"],
        categories=ATTACK_ORDER,
        ordered=True,
    )

    coverage = coverage.sort_values(
        ["metric", "attack"]
    ).reset_index(drop=True)

    print("\nCoverage and distributions\n")
    print(
        coverage.to_string(
            index=False,
            float_format=lambda value: f"{value:.2f}",
        )
    )

    return coverage


# ------------------------------------------------------------
# Four-panel vulnerability plot
# ------------------------------------------------------------

def plot_vulnerability_summary(summary, save=True):

    # Restrained blue-grey palette with a warm median accent.
    POINT_COLOR = "#6F9DBB"    # Individual target detectors
    TAIL_COLOR = "#566573"     # 10thâ€“90th percentile
    IQR_COLOR = "#263746"      # Interquartile range
    MEDIAN_COLOR = "#D55E00"   # Median
    GRID_COLOR = "#E1E6E9"     # Grid lines
    TEXT_COLOR = "#2B2B2B"     # Labels and titles
    SPINE_COLOR = "#AAB4BA"    # Bottom axis line

    sns.set_theme(
        style="whitegrid",
        context="paper",
        font_scale=1.0,
        rc={
            "text.color": TEXT_COLOR,
            "axes.labelcolor": TEXT_COLOR,
            "axes.titlecolor": TEXT_COLOR,
            "xtick.color": TEXT_COLOR,
            "ytick.color": TEXT_COLOR,
        },
    )

    fig, axes = plt.subplots(
        4,
        1,
        figsize=(3.45, 6.8),
        sharex=True,
        gridspec_kw={
            "height_ratios": [2, 2, 3, 3],
            "hspace": 0.50,
        },
    )

    def draw_distribution_row(ax, values, y, rng):
        """
        Points: individual target detectors
        Thin interval: 10thâ€“90th percentiles
        Thick interval: interquartile range
        Diamond: median
        """
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]

        if values.size == 0:
            return

        # Deterministic vertical jitter.
        jitter = rng.uniform(
            low=-0.13,
            high=0.13,
            size=values.size,
        )

        # Individual target detectors.
        ax.scatter(
            values,
            np.full(values.size, y) + jitter,
            s=9,
            color=POINT_COLOR,
            alpha=0.32,
            edgecolors="none",
            clip_on=False,
            zorder=2,
        )

        q10, q25, median, q75, q90 = np.percentile(
            values,
            [10, 25, 50, 75, 90],
        )

        # 10thâ€“90th percentile interval.
        ax.hlines(
            y=y,
            xmin=q10,
            xmax=q90,
            color=TAIL_COLOR,
            linewidth=1.15,
            zorder=3,
        )

        ax.vlines(
            x=[q10, q90],
            ymin=y - 0.055,
            ymax=y + 0.055,
            color=TAIL_COLOR,
            linewidth=1.0,
            zorder=3,
        )

        # Interquartile range.
        ax.hlines(
            y=y,
            xmin=q25,
            xmax=q75,
            color=IQR_COLOR,
            linewidth=3.4,
            zorder=4,
        )

        # Median marker.
        ax.scatter(
            median,
            y,
            marker="D",
            s=25,
            facecolor=MEDIAN_COLOR,
            edgecolor="white",
            linewidth=0.8,
            clip_on=False,
            zorder=5,
        )

    for panel_index, (ax, metric) in enumerate(
        zip(axes, METRIC_ORDER)
    ):
        panel = summary[
            summary["metric"].eq(metric)
        ].copy()

        if panel.empty:
            raise ValueError(
                f"No observations available for {metric}."
            )

        attacks = [
            attack
            for attack in ATTACK_ORDER
            if attack in panel["attack"].unique()
        ]

        # Fixed seed makes the jitter reproducible.
        rng = np.random.default_rng(
            1800 + panel_index
        )

        for y_position, attack in enumerate(attacks):
            values = panel.loc[
                panel["attack"].eq(attack),
                "value_pct",
            ].to_numpy()

            draw_distribution_row(
                ax=ax,
                values=values,
                y=y_position,
                rng=rng,
            )

        panel_letter = chr(
            ord("a") + panel_index
        )

        ax.set_title(
            f"({panel_letter}) {metric}",
            loc="left",
            fontsize=9.0,
            fontweight="semibold",
            color=TEXT_COLOR,
            pad=6,
        )

        ax.set_xlabel("")
        ax.set_ylabel("")

        ax.set_yticks(range(len(attacks)))
        ax.set_yticklabels(
            [
                ATTACK_LABELS[attack]
                for attack in attacks
            ],
            fontsize=8.2,
            color=TEXT_COLOR,
        )

        # Place the first attack at the top.
        ax.set_ylim(
            len(attacks) - 0.5,
            -0.5,
        )

        # Prevent boundary observations from being clipped.
        ax.set_xlim(-3, 103)
        ax.set_xticks([0, 25, 50, 75, 100])

        ax.tick_params(
            axis="x",
            labelsize=7.8,
            length=2.5,
            pad=2,
            labelbottom=True,
            colors=TEXT_COLOR,
        )
        ax.tick_params(
            axis="y",
            length=0,
            pad=6,
            colors=TEXT_COLOR,
        )

        ax.set_axisbelow(True)
        ax.grid(
            axis="x",
            color=GRID_COLOR,
            alpha=0.85,
            linewidth=0.55,
        )
        ax.grid(
            axis="y",
            visible=False,
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_visible(False)
        ax.spines["bottom"].set_color(SPINE_COLOR)
        ax.spines["bottom"].set_linewidth(0.7)

    axes[-1].set_xlabel(
        "Attack success rate (%)",
        fontsize=8.5,
        color=TEXT_COLOR,
        labelpad=5,
    )

    fig.subplots_adjust(
        top=0.985,
        bottom=0.075,
        left=0.225,
        right=0.97,
        hspace=0.50,
    )

    if save:
        OUTPUT_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        pdf_path = OUTPUT_STEM.with_suffix(".pdf")
        png_path = OUTPUT_STEM.with_suffix(".png")

        fig.savefig(
            pdf_path,
            bbox_inches="tight",
        )
        fig.savefig(
            png_path,
            dpi=600,
            bbox_inches="tight",
        )

        print(f"Saved: {pdf_path}")
        print(f"Saved: {png_path}")

    plt.show()
    return fig

# ------------------------------------------------------------
# Run
# ------------------------------------------------------------

print("Authoritative pairwise inputs")
print(f"  AA:     {PAIRWISE_PATHS['aa'].name}")
print(f"  CW-EOT: {PAIRWISE_PATHS['cw_eot'].name}")

oracle_files = discover_oracle_files(
    RESULTS_DIR
)

model_summary = build_summary(
    PAIRWISE_DATA,
    oracle_files,
)

coverage_summary = print_coverage(
    model_summary
)

if SAVE_FIGURE:
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    summary_path = (
        OUTPUT_DIR
        / "model_vulnerability_summary.csv"
    )

    coverage_path = (
        OUTPUT_DIR
        / "model_vulnerability_coverage.csv"
    )

    model_summary.to_csv(
        summary_path,
        index=False,
    )

    coverage_summary.to_csv(
        coverage_path,
        index=False,
    )

    print(f"\nSaved: {summary_path}")
    print(f"Saved: {coverage_path}")

fig = plot_vulnerability_summary(
    model_summary,
    save=SAVE_FIGURE,
)

## Core Transfer Analysis

Define preparation, diagnostics, summaries, transfer matrices, matched tests, and output-saving helpers.


In [ ]:
import pandas as pd
from transfer_analysis_node_jackknife import run_transfer_analysis

df_aa = df_aa.copy()
df_eot = df_eot.copy()

df_aa["attack_type"] = "aa"
df_eot["attack_type"] = "cw_eot"

results = run_transfer_analysis(
    df_transfer=pd.concat([df_aa, df_eot], ignore_index=True),
    out_dir=str(STATISTICS_DIR),
    alpha=0.05,
    exclude_clip=True,
)

primary_results = results["stats_summary"].query(
    "primary_holm_family"
)

display(primary_results[[
    "attack_type",
    "test_name",
    "mean_diff",
    "se_node_jackknife",
    "ci_low",
    "ci_high",
    "p_node_jackknife",
    "p_holm",
    "reject_holm",
    "most_influential_node",
]])


primary_results["holm_family_size"].unique()
# Must return: array([19])

print(len(primary_results))
# Must return: 19

## Group-Level Transfer Heatmaps


In [ ]:
# ============================================================
# Combined group-level black-box transfer heatmaps
# Paper-ready, full-width version
# ============================================================
# Produces one 2 x 4 figure:
#
# Rows:
#   - AA
#   - CW-EOT
#
# Columns:
#   - Architecture family
#   - Exact backbone
#   - Pretraining
#   - Training family
#
# Values:
#   - mean black-box ASR (%)
#   - white-box source-target pairs excluded
#   - CLIP excluded from controlled comparisons
#   - architecture-family panel additionally excludes pairs
#     sharing the exact same backbone
#
# Requires:
#   - df_aa
#   - df_eot
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

from transfer_analysis_node_jackknife import prepare_transfer_df


# ------------------------------------------------------------
# Orders and labels
# ------------------------------------------------------------

ATTACK_ORDER = ["aa", "cw_eot"]

ARCH_FAMILY_ORDER = ["CNN", "Transformer"]

BACKBONE_ORDER = [
    "resnet34",
    "xception",
    "efficientnetb4",
    "deit",
    "vit",
    "swin",
]

PRETRAIN_ORDER = ["fr", "imgnet"]

TRAIN_FAMILY_ORDER = ["ALL", "FS", "FR", "EFS", "FE"]

PRETRAIN_LABELS = {
    "fr": "FaceRec.",
    "imgnet": "ImageNet",
}

BACKBONE_LABELS = {
    "resnet34": "ResNet34",
    "xception": "Xception",
    "efficientnetb4": "EffNet-B4",
    "deit": "DeiT-S",
    "vit": "ViT-B/16",
    "swin": "Swin-T",
}


# ------------------------------------------------------------
# Colormap
# ------------------------------------------------------------

def make_asr_cmap():
    """
    Sequential ASR colormap suitable for ordered non-negative values.
    """
    return sns.color_palette("YlOrRd", as_cmap=True)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def infer_arch_family_from_backbone(backbone):
    b = str(backbone).lower()

    if b in ["resnet34", "xception", "efficientnetb4"]:
        return "CNN"

    if b in ["deit", "vit", "swin"]:
        return "Transformer"

    if b == "clip":
        return "CLIP"

    return "Other"


def _ordered_values(values, preferred_order):
    values = list(
        pd.Series(values)
        .dropna()
        .unique()
    )

    ordered = [
        value
        for value in preferred_order
        if value in values
    ]

    ordered += sorted([
        value
        for value in values
        if value not in ordered
    ])

    return ordered


def _label_matrix(matrix, label_map=None):
    if label_map is None:
        return matrix

    labeled = matrix.copy()

    labeled.index = [
        label_map.get(value, value)
        for value in labeled.index
    ]

    labeled.columns = [
        label_map.get(value, value)
        for value in labeled.columns
    ]

    return labeled


def _draw_diagonal_boxes(
    ax,
    matrix_to_plot,
    linewidth=1.25,
):
    """
    Highlight same-group transfer cells with a thin dark outline.
    """

    row_labels = list(matrix_to_plot.index)
    col_labels = list(matrix_to_plot.columns)

    for i, row_label in enumerate(row_labels):
        for j, col_label in enumerate(col_labels):
            if str(row_label) == str(col_label):
                ax.add_patch(
                    Rectangle(
                        (j, i),
                        1,
                        1,
                        fill=False,
                        edgecolor="0.15",
                        linewidth=linewidth,
                    )
                )


# ------------------------------------------------------------
# Data preparation
# ------------------------------------------------------------

def prepare_grouped_transfer_df_per_attack(
    df_aa,
    df_eot,
    exclude_clip=True,
):
    """
    Prepare black-box transfer dataframe for per-attack grouped
    heatmaps.

    Adds:
      - origin_arch_family
      - target_arch_family
      - same_arch_family

    Does not aggregate across attacks.
    Keeps AA and CW-EOT as separate rows.
    """

    parts = [
        prepare_transfer_df(
            df_aa,
            attack_type="aa",
        ),
        prepare_transfer_df(
            df_eot,
            attack_type="cw_eot",
        ),
    ]

    d = pd.concat(
        parts,
        ignore_index=True,
    ).copy()

    required_cols = [
        "attack_type",
        "origin_model",
        "target_model",
        "asr",
        "is_blackbox",
        "origin_pretrain",
        "target_pretrain",
        "origin_backbone",
        "target_backbone",
        "origin_train_family",
        "target_train_family",
    ]

    missing_cols = [
        col
        for col in required_cols
        if col not in d.columns
    ]

    if missing_cols:
        raise ValueError(
            "Missing required columns from "
            "prepare_transfer_df output: "
            f"{missing_cols}"
        )

    d["asr"] = pd.to_numeric(
        d["asr"],
        errors="coerce",
    )

    # Black-box only, valid ASR only.
    d = d.loc[
        d["is_blackbox"] & d["asr"].notna()
    ].copy()

    # Architecture family.
    d["origin_arch_family"] = (
        d["origin_backbone"]
        .apply(infer_arch_family_from_backbone)
    )

    d["target_arch_family"] = (
        d["target_backbone"]
        .apply(infer_arch_family_from_backbone)
    )

    d["same_arch_family"] = (
        d["origin_arch_family"]
        .eq(d["target_arch_family"])
    )

    # Remove CLIP completely. Check both parsed factors and
    # original model keys so naming variations cannot leak into
    # the controlled panels.
    if exclude_clip:
        clip_cols = [
            "origin_model",
            "target_model",
            "origin_pretrain",
            "target_pretrain",
            "origin_backbone",
            "target_backbone",
            "origin_arch_family",
            "target_arch_family",
        ]

        contains_clip = pd.Series(
            False,
            index=d.index,
        )

        for col in clip_cols:
            contains_clip |= (
                d[col]
                .astype(str)
                .str.contains(
                    "clip",
                    case=False,
                    na=False,
                )
            )

        d = d.loc[
            ~contains_clip
        ].copy()

    # Remove unknown/other architecture families.
    d = d.loc[
        d["origin_arch_family"].isin(
            ARCH_FAMILY_ORDER
        )
        & d["target_arch_family"].isin(
            ARCH_FAMILY_ORDER
        )
    ].copy()

    return d


def make_group_matrix(
    df,
    target_col,
    origin_col,
    target_order=None,
    origin_order=None,
):
    """
    Make the mean-ASR matrix and corresponding count matrix.

    Returns:
        matrix: ASR values in percentage points
        counts: number of black-box source-target rows per cell
    """

    matrix = (
        df.pivot_table(
            index=target_col,
            columns=origin_col,
            values="asr",
            aggfunc="mean",
        )
        * 100
    )

    counts = df.pivot_table(
        index=target_col,
        columns=origin_col,
        values="asr",
        aggfunc="count",
        fill_value=0,
    )

    target_order = _ordered_values(
        matrix.index,
        target_order or list(matrix.index),
    )

    origin_order = _ordered_values(
        matrix.columns,
        origin_order or list(matrix.columns),
    )

    matrix = matrix.reindex(
        index=target_order,
        columns=origin_order,
    )

    counts = counts.reindex(
        index=target_order,
        columns=origin_order,
    )

    return matrix, counts


def summarize_group_matrix(matrix):
    """
    Row mean = target vulnerability.
    Column mean = source transferability.
    """

    row_summary = pd.DataFrame({
        "group": matrix.index,
        "role": "target",
        "mean_asr_pct": (
            matrix
            .mean(axis=1, skipna=True)
            .values
        ),
    })

    col_summary = pd.DataFrame({
        "group": matrix.columns,
        "role": "origin",
        "mean_asr_pct": (
            matrix
            .mean(axis=0, skipna=True)
            .values
        ),
    })

    return pd.concat(
        [row_summary, col_summary],
        ignore_index=True,
    )


# ------------------------------------------------------------
# Plotting
# ------------------------------------------------------------

def plot_single_group_heatmap(
    ax,
    matrix,
    title,
    cmap,
    value_vmin=0,
    value_vmax=50,
    label_map=None,
    highlight_diagonal=True,
    annot_fontsize=8,
    annot_decimals=0,
    show_xticklabels=True,
    xtick_rotation=35,
    box_aspect=0.78,
):
    """
    Plot one compact heatmap with contrast-aware annotations.

    ``box_aspect`` is the height-to-width ratio of the complete
    panel. A value below 1 makes the cells slightly wider than tall.
    """

    matrix_to_plot = _label_matrix(
        matrix,
        label_map,
    )

    mask = matrix_to_plot.isna()

    norm = Normalize(
        vmin=value_vmin,
        vmax=value_vmax,
    )

    sns.heatmap(
        matrix_to_plot,
        ax=ax,
        annot=False,
        mask=mask,
        cmap=cmap,
        vmin=value_vmin,
        vmax=value_vmax,
        cbar=False,
        linewidths=0.55,
        linecolor="white",
        square=False,
    )

    ax.set_box_aspect(box_aspect)

    # Choose black or white annotation text according to the
    # rendered cell luminance.
    for i, row_label in enumerate(
        matrix_to_plot.index
    ):
        for j, col_label in enumerate(
            matrix_to_plot.columns
        ):
            value = matrix_to_plot.loc[
                row_label,
                col_label,
            ]

            if pd.isna(value):
                continue

            red, green, blue, _ = cmap(
                norm(value)
            )

            luminance = (
                0.2126 * red
                + 0.7152 * green
                + 0.0722 * blue
            )

            text_color = (
                "black"
                if luminance >= 0.52
                else "white"
            )

            ax.text(
                j + 0.5,
                i + 0.5,
                f"{value:.{annot_decimals}f}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=annot_fontsize,
            )

    if highlight_diagonal:
        _draw_diagonal_boxes(
            ax,
            matrix_to_plot,
            linewidth=0.9,
        )

    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")

    if show_xticklabels:
        ax.set_xticklabels(
            list(matrix_to_plot.columns),
            rotation=xtick_rotation,
            ha="right",
            rotation_mode="anchor",
            fontsize=6.6,
        )
    else:
        ax.tick_params(
            axis="x",
            labelbottom=False,
        )

    ax.set_yticklabels(
        list(matrix_to_plot.index),
        rotation=0,
        fontsize=6.6,
    )

    ax.tick_params(
        axis="both",
        length=0,
        pad=1.5,
    )

    return ax


def plot_factor_transfer_figure(
    df_grouped,
    specs,
    value_vmin=0,
    value_vmax=40,
    figsize=(7.15, 3.85),
    width_ratios=None,
    save_path=None,
    raster_dpi=1200,
    highlight_diagonal=True,
    panel_box_aspect=1.0,
):
    """
    Plot the complete 2 x 4 paper figure.

    Rows:
      - AA
      - CW-EOT

    Columns:
      - factor matrices defined in specs
    """

    paper_rc = {
        "font.family": "sans-serif",
        "font.sans-serif": [
            "Arial",
            "Helvetica",
            "DejaVu Sans",
        ],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.6,
    }

    with plt.rc_context(paper_rc):
        cmap = make_asr_cmap()

        n_rows = len(ATTACK_ORDER)
        n_cols = len(specs)

        if width_ratios is None:
            width_ratios = [1.0] * n_cols

        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=figsize,
            gridspec_kw={
                "width_ratios": width_ratios,
            },
        )

        if n_rows == 1:
            axes = np.array([axes])

        if n_cols == 1:
            axes = axes.reshape(
                n_rows,
                1,
            )

        matrices = {}
        counts = {}
        summaries = {}

        for row_idx, attack_type in enumerate(
            ATTACK_ORDER
        ):
            attack_df = df_grouped.loc[
                df_grouped["attack_type"].eq(
                    attack_type
                )
            ].copy()

            for col_idx, spec in enumerate(specs):
                ax = axes[row_idx, col_idx]

                # Start each panel from the complete attack-specific
                # black-box dataframe.
                panel_df = attack_df.copy()

                # For the architecture-family panel only, remove
                # source-target pairs sharing the same exact backbone.
                #
                # The remaining diagonal cells therefore mean:
                # same CNN/Transformer family, different backbone.
                if spec.get(
                    "exclude_same_backbone",
                    False,
                ):
                    panel_df = panel_df.loc[
                        ~panel_df[
                            "origin_backbone"
                        ].eq(
                            panel_df[
                                "target_backbone"
                            ]
                        )
                    ].copy()

                matrix, count_matrix = (
                    make_group_matrix(
                        panel_df,
                        target_col=(
                            spec["target_col"]
                        ),
                        origin_col=(
                            spec["origin_col"]
                        ),
                        target_order=(
                            spec["target_order"]
                        ),
                        origin_order=(
                            spec["origin_order"]
                        ),
                    )
                )

                key = (
                    f"{attack_type}__"
                    f"{spec['name']}"
                )

                matrices[key] = matrix
                counts[key] = count_matrix
                summaries[key] = (
                    summarize_group_matrix(matrix)
                )

                plot_single_group_heatmap(
                    ax=ax,
                    matrix=matrix,
                    title="",
                    cmap=cmap,
                    value_vmin=value_vmin,
                    value_vmax=value_vmax,
                    label_map=spec.get(
                        "label_map"
                    ),
                    highlight_diagonal=(
                        highlight_diagonal
                    ),
                    annot_fontsize=spec.get(
                        "annot_fontsize",
                        6.5,
                    ),
                    annot_decimals=spec.get(
                        "annot_decimals",
                        0,
                    ),
                    show_xticklabels=(
                        row_idx == n_rows - 1
                    ),
                    xtick_rotation=spec.get(
                        "xtick_rotation",
                        35,
                    ),
                    box_aspect=spec.get(
                        "box_aspect",
                        panel_box_aspect,
                    ),
                )

        fig.subplots_adjust(
            left=0.095,
            right=0.915,
            top=0.90,
            bottom=0.20,
            wspace=0.48,
            hspace=0.26,
        )

        # Column titles share one baseline.
        fig.canvas.draw()

        for col_idx, spec in enumerate(specs):
            bbox = axes[
                0,
                col_idx,
            ].get_position()

            x_center = 0.5 * (
                bbox.x0 + bbox.x1
            )

            fig.text(
                x_center,
                0.965,
                (
                    f"({chr(97 + col_idx)}) "
                    f"{spec['name']}"
                ),
                va="center",
                ha="center",
                fontsize=8.2,
            )

        fig.text(
            0.012,
            0.50,
            "Target group",
            rotation=90,
            va="center",
            ha="center",
            fontsize=7.2,
        )

        fig.text(
            0.51,
            0.035,
            "Source group",
            va="center",
            ha="center",
            fontsize=7.2,
        )

        # One shared scale for all eight matrices.
        norm = Normalize(
            vmin=value_vmin,
            vmax=value_vmax,
        )

        sm = ScalarMappable(
            norm=norm,
            cmap=cmap,
        )
        sm.set_array([])

        cbar_ax = fig.add_axes([
            0.94,
            0.235,
            0.014,
            0.53,
        ])

        cbar = fig.colorbar(
            sm,
            cax=cbar_ax,
            ticks=np.arange(
                value_vmin,
                value_vmax + 1,
                10,
            ),
        )

        cbar.set_label(
            "Mean black-box ASR (%)",
            fontsize=7.2,
            labelpad=4,
        )

        cbar.ax.tick_params(
            labelsize=6.5,
            length=2,
            width=0.5,
        )

        cbar.outline.set_linewidth(0.5)

        if save_path is not None:
            output_path = Path(save_path)

            if output_path.suffix.lower() == ".pdf":
                # Keep a vector PDF for the paper and also create
                # an ultra-high-resolution raster version.
                fig.savefig(
                    output_path,
                    bbox_inches=None,
                )

                fig.savefig(
                    output_path.with_suffix(".png"),
                    dpi=raster_dpi,
                    bbox_inches=None,
                )
            else:
                fig.savefig(
                    output_path,
                    dpi=raster_dpi,
                    bbox_inches=None,
                )

        plt.show()

    return {
        "matrices": matrices,
        "counts": counts,
        "summaries": summaries,
    }


# ============================================================
# Prepare per-attack grouped transfer dataframe
# ============================================================

grouped_transfer_per_attack = (
    prepare_grouped_transfer_df_per_attack(
        df_aa=df_aa,
        df_eot=df_eot,
        exclude_clip=True,
    )
)


# ============================================================
# Combined figure specifications
# ============================================================

COMBINED_FACTOR_SPECS = [
    {
        "name": "Architecture family",
        "target_col": "target_arch_family",
        "origin_col": "origin_arch_family",
        "target_order": ARCH_FAMILY_ORDER,
        "origin_order": ARCH_FAMILY_ORDER,
        "label_map": None,

        # Only this panel excludes exact shared-backbone pairs.
        "exclude_same_backbone": True,

        "annot_fontsize": 6.8,
        "annot_decimals": 0,
        "xtick_rotation": 35,
        "box_aspect": 1.00,
    },
    {
        "name": "Backbone",
        "target_col": "target_backbone",
        "origin_col": "origin_backbone",
        "target_order": BACKBONE_ORDER,
        "origin_order": BACKBONE_ORDER,
        "label_map": BACKBONE_LABELS,
        "annot_fontsize": 6.2,
        "annot_decimals": 0,
        "xtick_rotation": 38,
        "box_aspect": 0.80,
    },
    {
        "name": "Pretraining",
        "target_col": "target_pretrain",
        "origin_col": "origin_pretrain",
        "target_order": PRETRAIN_ORDER,
        "origin_order": PRETRAIN_ORDER,
        "label_map": PRETRAIN_LABELS,
        "annot_fontsize": 6.8,
        "annot_decimals": 0,
        "xtick_rotation": 38,
        "box_aspect": 1.00,
    },
    {
        "name": "Training data",
        "target_col": "target_train_family",
        "origin_col": "origin_train_family",
        "target_order": TRAIN_FAMILY_ORDER,
        "origin_order": TRAIN_FAMILY_ORDER,
        "label_map": None,
        "annot_fontsize": 6.3,
        "annot_decimals": 0,
        "xtick_rotation": 35,
        "box_aspect": 0.87,
    },
]


# ============================================================
# Generate Figure 3
# ============================================================

combined_factor_heatmaps = (
    plot_factor_transfer_figure(
        grouped_transfer_per_attack,
        specs=COMBINED_FACTOR_SPECS,
        value_vmin=0,
        value_vmax=40,
        figsize=(7.15, 3.85),
        width_ratios=[
            1.00,
            1.25,
            1.00,
            1.15,
        ],
        save_path=(
            FIGURES_DIR / "group_transfer_factors_combined.pdf"
        ),
        raster_dpi=1200,
        highlight_diagonal=True,
        panel_box_aspect=1.0,
    )
)

## Sensitivity Analysis

In [ ]:
import pandas as pd
from transfer_analysis_node_jackknife import run_all_matched_tests

prepared_df = results["df"].copy() #FROM RUNNING CORE TRANSFER ANALYSIS
def run_leave_one_level_out(
    df,
    origin_col,
    target_col,
    factor_name,
    alpha=0.05,
):
    outputs = []

    levels = sorted(
        set(df[origin_col].dropna().astype(str))
        | set(df[target_col].dropna().astype(str))
    )

    for i, level in enumerate(levels, start=1):
        print(
            f"{factor_name}: {i}/{len(levels)}  removing {level}"
        )

        # Remove the whole level from both roles and both attacks
        reduced_df = df[
            df[origin_col].astype(str).ne(level)
            & df[target_col].astype(str).ne(level)
        ].copy()

        stats_summary, _, _ = run_all_matched_tests(
            reduced_df,
            alpha=alpha,
        )

        primary = stats_summary.query(
            "primary_holm_family"
        ).copy()

        primary["omission_factor"] = factor_name
        primary["omitted_level"] = level
        primary["num_rows_remaining"] = len(reduced_df)

        outputs.append(primary)

    return pd.concat(outputs, ignore_index=True)


# Omit one backbone architecture at a time
loo_backbone = run_leave_one_level_out(
    prepared_df,
    origin_col="origin_backbone",
    target_col="target_backbone",
    factor_name="backbone",
)

# Omit one training family at a time
loo_train_family = run_leave_one_level_out(
    prepared_df,
    origin_col="origin_train_family",
    target_col="target_train_family",
    factor_name="training_family",
)

loo_grouped = pd.concat(
    [loo_backbone, loo_train_family],
    ignore_index=True,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

key_cols = ["scope", "attack_type", "test_name"]

# Full-data estimates
full = (
    results["stats_summary"]
    .query("primary_holm_family")
    [key_cols + [
        "mean_diff",
        "ci_low",
        "ci_high",
        "p_holm",
        "reject_holm",
        "holm_family_size",
    ]]
    .rename(columns={
        "mean_diff": "full_effect",
        "ci_low": "full_ci_low",
        "ci_high": "full_ci_high",
        "p_holm": "full_p_holm",
        "reject_holm": "full_reject_holm",
    })
)

assert len(full) == 19
assert set(full["holm_family_size"].dropna()) == {19}

compatibility_tests = [
    "same_backbone",
    "same_pretrain",
    "same_train_family_excl_ALL",
    "same_arch_family_excl_same_backbone",
]

interaction_tests = [
    f"attack_interaction_{x}"
    for x in compatibility_tests
]

central_mask = (
    (
        loo_grouped["scope"].eq("per_attack")
        & loo_grouped["test_name"].isin(compatibility_tests)
    )
    | (
        loo_grouped["scope"].eq("attack_interaction")
        & loo_grouped["test_name"].isin(interaction_tests)
    )
    | (
        loo_grouped["scope"].eq("cross_attack")
        & loo_grouped["test_name"].eq("attack_cw_eot_vs_aa")
    )
)

central_loo = (
    loo_grouped.loc[central_mask]
    .merge(
        full,
        on=key_cols,
        how="left",
        validate="many_to_one",
    )
    .copy()
)

assert central_loo["full_effect"].notna().all()

central_loo["change_from_full"] = (
    central_loo["mean_diff"] - central_loo["full_effect"]
)
central_loo["abs_change_from_full"] = (
    central_loo["change_from_full"].abs()
)
central_loo["sign_preserved"] = (
    np.sign(central_loo["mean_diff"])
    == np.sign(central_loo["full_effect"])
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D


# ============================================================
# Result definitions and plotting order
# ============================================================

compatibility_tests = [
    "same_backbone",
    "same_pretrain",
    "same_train_family_excl_ALL",
    "same_arch_family_excl_same_backbone",
]

attack_order = ["aa", "cw_eot"]

interaction_tests = [
    f"attack_interaction_{test}"
    for test in compatibility_tests
]

ordered_keys = (
    [
        f"{test}|{attack}"
        for test in compatibility_tests
        for attack in attack_order
    ]
    + interaction_tests
    + ["attack_cw_eot_vs_aa"]
)

base_labels = {
    "same_backbone":
        "Shared backbone",
    "same_pretrain":
        "Shared pretraining",
    "same_train_family_excl_ALL":
        "Shared training dataset",
    "same_arch_family_excl_same_backbone":
        "Shared architecture family",
}


# ============================================================
# Create plotting identifiers
# ============================================================

def make_display_key(row):
    if row["scope"] == "per_attack":
        return (
            f'{row["test_name"]}|'
            f'{row["attack_type"]}'
        )

    return row["test_name"]


def make_result_label(row):
    if row["scope"] == "per_attack":
        attack_label = {
            "aa": "AA",
            "cw_eot": "CW-EOT",
        }.get(
            row["attack_type"],
            str(row["attack_type"]),
        )

        return (
            f'{base_labels[row["test_name"]]}'
            f' - {attack_label}'
        )

    if row["scope"] == "attack_interaction":
        base_test = row["test_name"].replace(
            "attack_interaction_",
            "",
        )

        return (
            f'{base_labels[base_test]}: '
            r'CW-EOT $-$ AA'
        )

    if row["test_name"] == "attack_cw_eot_vs_aa":
        return (
            r'Overall attack effect: '
            r'CW-EOT $-$ AA'
        )

    return str(row["test_name"])


central_loo = central_loo.copy()

central_loo["display_key"] = central_loo.apply(
    make_display_key,
    axis=1,
)

central_loo["result_label"] = central_loo.apply(
    make_result_label,
    axis=1,
)

order_map = {
    key: position
    for position, key in enumerate(ordered_keys)
}

central_loo["display_order"] = (
    central_loo["display_key"].map(order_map)
)


# ============================================================
# Validate the plotting structure
# ============================================================

required_metadata_cols = [
    "display_key",
    "result_label",
    "display_order",
    "omission_factor",
]

missing_metadata = [
    col
    for col in required_metadata_cols
    if col not in central_loo.columns
]

if missing_metadata:
    raise KeyError(
        "Missing plotting metadata columns: "
        + ", ".join(missing_metadata)
    )

available_keys = set(
    central_loo["display_key"].dropna()
)

missing_keys = [
    key
    for key in ordered_keys
    if key not in available_keys
]

if missing_keys:
    print(
        "Warning: the following requested results "
        "are absent from central_loo:"
    )

    for key in missing_keys:
        print(f"  - {key}")


# ============================================================
# Prepare numerical columns
# ============================================================

plot_df = central_loo.copy()

# Change these only if the omission-specific CI columns
# have different names.
effect_col = "mean_diff"
ci_low_col = "ci_low"
ci_high_col = "ci_high"

required_effect_cols = [
    effect_col,
    ci_low_col,
    ci_high_col,
    "full_effect",
    "full_ci_low",
    "full_ci_high",
]

missing_effect_cols = [
    col
    for col in required_effect_cols
    if col not in plot_df.columns
]

if missing_effect_cols:
    raise KeyError(
        "Missing columns required for plotting CIs: "
        + ", ".join(missing_effect_cols)
    )

# Convert effects and confidence limits to percentage points.
plot_df[required_effect_cols] = (
    plot_df[required_effect_cols]
    .apply(pd.to_numeric, errors="coerce")
    * 100
)


# ============================================================
# Panel definitions
# ============================================================

panel_specs = [
    (
        "Attack-specific compatibility effects",
        ordered_keys[:8],
    ),
    (
        r"Attack interactions: CW-EOT $-$ AA",
        ordered_keys[8:12],
    ),
    (
        r"Overall attack contrast: CW-EOT $-$ AA",
        ordered_keys[12:],
    ),
]

styles = {
    "backbone": {
        "color": "#2878B5",
        "marker": "o",
        "offset_range": (-0.34, -0.08),
    },
    "training_family": {
        "color": "#E67E22",
        "marker": "^",
        "offset_range": (0.08, 0.34),
    },
}

# Used only to give omissions a consistent vertical ordering.
possible_level_cols = [
    "omitted_level",
    "omission_level",
    "omitted_value",
    "omitted_group",
]

omitted_level_col = next(
    (
        col
        for col in possible_level_cols
        if col in plot_df.columns
    ),
    None,
)


# ============================================================
# Plot
# ============================================================

fig, axes = plt.subplots(
    3,
    1,
    figsize=(9.5, 13),
    gridspec_kw={
        "height_ratios": [8, 4, 1.7],
    },
)

for ax, (title, panel_keys) in zip(
    axes,
    panel_specs,
):
    panel = plot_df[
        plot_df["display_key"].isin(panel_keys)
    ].copy()

    label_lookup = (
        panel[
            ["display_key", "result_label"]
        ]
        .drop_duplicates(
            subset=["display_key"]
        )
        .set_index("display_key")["result_label"]
        .to_dict()
    )

    for y, key in enumerate(panel_keys):
        d = panel[
            panel["display_key"].eq(key)
        ].copy()

        if d.empty:
            continue

        full_values = (
            d[
                [
                    "full_effect",
                    "full_ci_low",
                    "full_ci_high",
                ]
            ]
            .dropna()
            .drop_duplicates()
        )

        if not full_values.empty:
            full_effect = (
                full_values["full_effect"].iloc[0]
            )
            full_ci_low = (
                full_values["full_ci_low"].iloc[0]
            )
            full_ci_high = (
                full_values["full_ci_high"].iloc[0]
            )

            # Full-data estimate and its CI.
            ax.errorbar(
                full_effect,
                y,
                xerr=np.array([
                    [full_effect - full_ci_low],
                    [full_ci_high - full_effect],
                ]),
                fmt="D",
                color="black",
                ecolor="black",
                markerfacecolor="black",
                markeredgecolor="black",
                markersize=5.5,
                elinewidth=1.5,
                capsize=3,
                capthick=1.2,
                linestyle="none",
                zorder=5,
            )

        # Each omission estimate and its recomputed CI.
        for factor, style in styles.items():
            factor_data = d[
                d["omission_factor"].eq(factor)
            ].copy()

            factor_data = factor_data.dropna(
                subset=[
                    effect_col,
                    ci_low_col,
                    ci_high_col,
                ]
            )

            if factor_data.empty:
                continue

            # Maintain the same vertical ordering across results.
            if omitted_level_col is not None:
                factor_data = factor_data.sort_values(
                    omitted_level_col,
                    kind="stable",
                )

            estimates = (
                factor_data[effect_col]
                .to_numpy(dtype=float)
            )

            ci_lows = (
                factor_data[ci_low_col]
                .to_numpy(dtype=float)
            )

            ci_highs = (
                factor_data[ci_high_col]
                .to_numpy(dtype=float)
            )

            low_offset, high_offset = (
                style["offset_range"]
            )

            if len(factor_data) == 1:
                offsets = np.array([
                    (low_offset + high_offset) / 2
                ])
            else:
                offsets = np.linspace(
                    low_offset,
                    high_offset,
                    len(factor_data),
                )

            y_positions = y + offsets

            ax.errorbar(
                estimates,
                y_positions,
                xerr=np.vstack([
                    estimates - ci_lows,
                    ci_highs - estimates,
                ]),
                fmt=style["marker"],
                color=style["color"],
                ecolor=style["color"],
                markerfacecolor=style["color"],
                markeredgecolor=style["color"],
                markersize=4.5,
                elinewidth=0.9,
                capsize=1.8,
                capthick=0.8,
                linestyle="none",
                alpha=0.80,
                zorder=3,
            )

    ax.axvline(
        0,
        color="0.45",
        linestyle="--",
        linewidth=1,
        zorder=1,
    )

    ax.set_yticks(
        range(len(panel_keys))
    )

    ax.set_yticklabels([
        label_lookup.get(key, key)
        for key in panel_keys
    ])

    # Leave room for vertically dodged intervals.
    ax.set_ylim(
        -0.48,
        len(panel_keys) - 0.52,
    )
    ax.invert_yaxis()

    ax.set_title(
        title,
        loc="left",
        fontsize=11,
    )

    ax.set_xlabel(
        "Difference in ASR "
        "(percentage points)"
    )

    ax.grid(
        axis="x",
        alpha=0.20,
        zorder=0,
    )


# ============================================================
# Legend
# ============================================================

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="D",
        color="black",
        markerfacecolor="black",
        markersize=5.5,
        linewidth=1.5,
        label="Full estimate and 95% CI",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color=styles["backbone"]["color"],
        markerfacecolor=styles["backbone"]["color"],
        markersize=4.5,
        linewidth=1,
        label=(
            "Leave-one-backbone-out estimates "
            "and 95% CIs"
        ),
    ),
    Line2D(
        [0],
        [0],
        marker="^",
        color=styles["training_family"]["color"],
        markerfacecolor=(
            styles["training_family"]["color"]
        ),
        markersize=4.5,
        linewidth=1,
        label=(
            "Leave-one-training-dataset-out "
            "estimates and 95% CIs"
        ),
    ),
]

axes[0].legend(
    handles=legend_handles,
    loc="best",
    frameon=False,
    fontsize=9,
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "grouped_loo_sensitivity_with_cis.pdf",
    bbox_inches="tight",
)

fig.savefig(
    FIGURES_DIR / "grouped_loo_sensitivity_with_cis.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
range_summary = (
    central_loo
    .groupby(
        ["display_key", "result_label", "omission_factor"],
        as_index=False,
    )
    .agg(
        minimum_effect=("mean_diff", "min"),
        maximum_effect=("mean_diff", "max"),
        valid_refits=("mean_diff", "count"),
    )
)

range_wide = range_summary.pivot(
    index=["display_key", "result_label"],
    columns="omission_factor",
    values=["minimum_effect", "maximum_effect"],
).reset_index()

# Flatten pivoted column names
range_wide.columns = [
    "_".join(
        str(part) for part in col if str(part)
    ).strip("_")
    if isinstance(col, tuple)
    else col
    for col in range_wide.columns
]

# Most influential omission across both factors
valid = central_loo.dropna(
    subset=["abs_change_from_full"]
).copy()

idx = (
    valid.groupby("display_key")
    ["abs_change_from_full"]
    .idxmax()
)

worst_case = (
    valid.loc[
        idx,
        [
            "display_key",
            "omission_factor",
            "omitted_level",
            "abs_change_from_full",
            "change_from_full",
        ],
    ]
    .rename(columns={
        "omission_factor": "worst_omission_type",
        "omitted_level": "worst_omitted_level",
        "abs_change_from_full": "maximum_abs_change",
        "change_from_full": "signed_change",
    })
)

full_for_table = (
    central_loo
    .sort_values("display_order")
    .groupby(
        ["display_key", "result_label"],
        as_index=False,
    )
    .agg(
        display_order=("display_order", "first"),
        full_effect=("full_effect", "first"),
        full_ci_low=("full_ci_low", "first"),
        full_ci_high=("full_ci_high", "first"),
        sign_always_preserved=("sign_preserved", "all"),
    )
)

annex_table = (
    full_for_table
    .merge(
        range_wide,
        on=["display_key", "result_label"],
        validate="one_to_one",
    )
    .merge(
        worst_case,
        on="display_key",
        validate="one_to_one",
    )
    .sort_values("display_order")
)

# Convert effects to percentage points
effect_cols = [
    "full_effect",
    "full_ci_low",
    "full_ci_high",
    "minimum_effect_backbone",
    "maximum_effect_backbone",
    "minimum_effect_training_family",
    "maximum_effect_training_family",
    "maximum_abs_change",
    "signed_change",
]

for col in effect_cols:
    if col in annex_table:
        annex_table[col] *= 100

# Compact printable fields
annex_table["full_effect_ci"] = annex_table.apply(
    lambda r:
        f'{r["full_effect"]:.2f} '
        f'[{r["full_ci_low"]:.2f}, '
        f'{r["full_ci_high"]:.2f}]',
    axis=1,
)

annex_table["backbone_omission_range"] = (
    annex_table.apply(
        lambda r:
            f'{r["minimum_effect_backbone"]:.2f} -'
            f'{r["maximum_effect_backbone"]:.2f}',
        axis=1,
    )
)

annex_table["training_family_omission_range"] = (
    annex_table.apply(
        lambda r:
            f'{r["minimum_effect_training_family"]:.2f} -'
            f'{r["maximum_effect_training_family"]:.2f}',
        axis=1,
    )
)

display_cols = [
    "result_label",
    "full_effect_ci",
    "backbone_omission_range",
    "training_family_omission_range",
    "maximum_abs_change",
    "worst_omission_type",
    "worst_omitted_level",
    "sign_always_preserved",
]

display(
    annex_table[display_cols]
    .rename(columns={
        "result_label": "Result",
        "full_effect_ci": "Full effect [95% CI]",
        "backbone_omission_range": "Backbone-out range",
        "training_family_omission_range":
            "Training-family-out range",
        "maximum_abs_change": "Maximum |change|",
        "worst_omission_type": "Omission type",
        "worst_omitted_level": "Most influential level",
        "sign_always_preserved": "Sign preserved",
    })
)